In [28]:
%load_ext autoreload
%autoreload 2
%reset -f

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers


import os
import sys
import pandas as pd
import geopandas as gpd
from pathlib import Path
from datetime import datetime, date, timedelta


# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers
from locallib.pandas import *

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.ingester.IngesterClass import Ingester
from lib.KPIHubConnection import *
from lib.query.bank import *

from datetime import date
from datetime import timedelta
class SurveySummaryIngester(Ingester):
    def __init__(self, arguments):
        super().__init__(arguments)
        self.table = KPI_SurveySummary

    def update_check(self):
        customer_name = self.customer_info['Name']
        customer_id = self.customer_info['CustomerId']
        customer_db = self.customer_info['DBLocation']

        self.Logger.info(f"Processing customer: {customer_name}")
        self.Logger.info(f"Getting reports from {self.update_window} to {self.current_date}")
        #Query the last report
        self.data['reports'] = Query(
            f"""
            SELECT ReportId, ReportDate, ReportArea, LastUpdated FROM KPI_ReportSummary
            WHERE CustomerId = '{self.customer_info['CustomerId']}'
            ORDER BY LastUpdated DESC
            """
        ).execute(KPIHub_Conn)

        self.data['survey_count'] = Query(
            f"""
            SELECT COUNT(*) as SurveyCount
            FROM KPI_SurveySummary
            WHERE ReportId IN (SELECT ReportId FROM KPI_ReportSummary WHERE CustomerId = '{customer_id}')
            """
        ).execute(KPIHub_Conn)

        if len(self.data['reports']) > 0:
            if (self.data['survey_count'].iloc[0]['SurveyCount']) > 0:
                self.check_flag = True
                self.starting_date = self.update_window
                self.Logger.info(f"Getting emissions from {self.update_window} to {self.current_date}")

            else:
                self.Logger.info(f"No emissions found, processing from start")
                self.starting_date = STARTING_DATE
                self.check_flag = True
        else:
            self.Logger.info(f"No reports found, skipping")
            self.check_flag = False

    def query_data(self):
        if self.check_flag:
            # Ensure the ReportDate values are in datetime format before comparison,
            # handling both with and without microseconds (mixed formats)
            self.data['reports']['ReportDate'] = pd.to_datetime(self.data['reports']['ReportDate'], format='mixed')

            # Handle potential issues with type mismatch when comparing datetimes
            # Coerce both sides to date for a robust comparison
            #reports_to_query = self.data['reports'][
            #    pd.to_datetime(self.data['reports']['ReportDate']).dt.date >= pd.to_datetime(self.starting_date).date()
            #]
            reports_to_query = self.data['reports'].copy()
            reports_to_query.sort_values(by='ReportDate', inplace=True)
            reports_to_query = reports_to_query.iloc[0:200]
   
            reports_to_query.db.set_query(query_surveys_table(report_table="#TempReports"))
            surveys = reports_to_query.db.execute(CONN_DICT[self.customer_info['DBLocation']], source_col = 'ReportId', temp_table_name = '#TempReports')
            surveys.db.set_query(query_segments_table(survey_table="#TempSurvey"))
            self.Logger.info(f"Reports from KPI_ReportSummary: {len(reports_to_query)}")
            self.Logger.info(f"Surveys from LSDB: {len(surveys)}")
            segments = surveys.db.execute(CONN_DICT[self.customer_info['DBLocation']], source_col = 'SurveyId', temp_table_name = '#TempSurvey')
            self.Logger.info(f"Segments from LSDB: {len(segments)}")


            # Set the starting time as a datetime object
            surveys['StartHour'] = pd.to_datetime(surveys['StartEpoch'], unit='s').dt.hour
            surveys['StartTime'] = pd.to_datetime(surveys['StartEpoch'], unit='s').dt.time
            surveys['EndHour'] = pd.to_datetime(surveys['EndEpoch'], unit='s').dt.hour
            surveys['EndTime'] = pd.to_datetime(surveys['EndEpoch'], unit='s').dt.time
            surveys['StartDay'] = pd.to_datetime(surveys['StartEpoch'], unit='s').dt.date
            surveys['EndDay'] = pd.to_datetime(surveys['EndEpoch'], unit='s').dt.date

            # Calculate the duration (in minutes) between StartEpoch and EndEpoch for each survey
            surveys['DurationMinutes'] = (
                surveys['EndEpoch'] - surveys['StartEpoch']
            ) / 60      

            # Prepare the report_gdf for report area lookup
            reports_to_query["geometry"] = gpd.GeoSeries.from_wkt(reports_to_query["ReportArea"])

            report_gdf = gpd.GeoDataFrame(
                reports_to_query,
                geometry="geometry",
                crs="EPSG:4326"
            )
            utm_crs = report_gdf.estimate_utm_crs()
            report_gdf = report_gdf.to_crs(utm_crs)
            report_gdf = report_gdf.set_index("ReportId")

            segments_gdf = gpd.GeoDataFrame(
                segments,
                geometry=gpd.GeoSeries.from_wkt(segments['Shape']),
                crs="EPSG:4326"
            )
            segments_gdf = segments_gdf.to_crs(utm_crs)
        
            survey_summary = surveys.apply(survey_summary_apply, axis=1)
            outputs =  []

            for idx, row in surveys.iterrows():
                self.Logger.File.info(f"Processing survey {row['SurveyId']} of {len(surveys)}")
                report_id  = row['ReportId']
                survey_id  = row['SurveyId']
                report_area_gdf = report_gdf.loc[[report_id]]
                segments_subset = segments_gdf[segments_gdf['SurveyId'] == survey_id]
                if segments_subset.empty or report_area_gdf.empty:
                    continue  # skip if no segments or report area is missing
                
                segments_in_report = gpd.overlay(
                    segments_subset,
                    report_area_gdf,
                    how='intersection',
                    keep_geom_type=False
                )

                if segments_in_report.empty:
                    continue  # skip if intersection is empty

                # Convert StartEpoch to datetime (time only, no date)
                segments_in_report['StartTime'] = pd.to_datetime(segments_in_report['StartEpoch'], unit='s').dt.time
                segments_in_report['StartDate'] = pd.to_datetime(segments_in_report['StartEpoch'], unit='s')
                segments_in_report['DayNight'] = segments_in_report['StartTime'].apply(lambda t: get_day_night(t, SUNRISE_TIME, SUNSET_TIME))
                segments_in_report['ActiveIdle'] = segments_in_report['CarSpeedMedian'].apply(lambda x: set_actie_idle(x, SPEED_THRESHOLD))

                output = {
                    'SurveyId': row['SurveyId'],
                    'ReportId': row['ReportId'],
                    'DaySegments': (segments_in_report['DayNight'] == 'Day').sum(),
                    'NightSegments': (segments_in_report['DayNight'] == 'Night').sum(),
                    'ActiveSegments': (segments_in_report['ActiveIdle'] == 'Active').sum(),
                    'IdleSegments': (segments_in_report['ActiveIdle'] == 'Idle').sum(),
                    'TotalSegments': len(segments_in_report),
                    'TotalKilometers': segments_in_report['LengthMeters'].sum() / 1000,
                    'DayKilometers': segments_in_report.loc[segments_in_report['DayNight'] == 'Day', 'LengthMeters'].sum() / 1000,
                    'NightKilometers': segments_in_report.loc[segments_in_report['DayNight'] == 'Night', 'LengthMeters'].sum() / 1000,
                    'SegmentDurationMinutes': segments_in_report['DurationSeconds'].sum() / 60,
                    'IdleTimeMinutes': segments_in_report.loc[segments_in_report['ActiveIdle'] == 'Idle', 'DurationSeconds'].sum() / 60,
                    'ActiveTimeMinutes': segments_in_report.loc[segments_in_report['ActiveIdle'] == 'Active', 'DurationSeconds'].sum() / 60,
                    'AvgSpeedKm': 3.6 * segments_in_report['CarSpeedMedian'].mean(),
            
                    'TotalSegmentsInSurvey': len(segments_subset),
                }
                outputs.append(output)

            output_df = pd.DataFrame(outputs)
            merged_df = pd.merge(
                survey_summary,
                output_df,
                left_on=["SurveyId", "ReportId"],
                right_on=["SurveyId", "ReportId"],
                how="inner"
            )
            merged_df['SegmentWeight'] = merged_df['TotalSegments'] / merged_df['TotalSegmentsInSurvey']
            merged_df['SurveyDurationMinutes'] = merged_df['SurveyRawDurationMinutes'] * merged_df['SegmentWeight']
            self.data['output'] = merged_df
            self.data['output']['LastUpdated'] = datetime.now()
        
        
    def sanity_check(self):
        super().sanity_check()
        #Get all the reports from the KPI_SurveySummary table
        df_surveys = Query(query = f"SELECT DISTINCT ReportId FROM KPI_SurveySummary WHERE ReportId IN (SELECT ReportId FROM KPI_ReportSummary WHERE CustomerId = '{self.customer_info['CustomerId']}')").execute(KPIHub_Conn)
        self.Logger.info(f"Total number of unique reports from KPI_SurveySummary: {len(df_surveys)}")

    def push_data(self):
        super().push_data(primary_key = ['SurveyId','ReportId'])
        self.Logger.info(f"Data pushed to {db_path}")

# Define a function to determine if survey is in 'day' or 'night'
def get_day_night(start_time, sunrise, sunset):
    # start_time should be a datetime.time object
    hour = start_time.hour
    if sunrise <= hour < sunset:
        return 'Day'
    else:
        return 'Night'

def set_actie_idle(speed, speed_threshold):
    if speed < speed_threshold:
        return 'Idle'
    else:
        return 'Active' 
def survey_summary_apply(row):
    return pd.Series({
        'SurveyId': row['SurveyId'],
        'SurveyorUnit': row['SurveyorUnit'],
        'SurveyRawDurationMinutes': row['DurationMinutes'],
        'ReportId': row['ReportId'],
        'StartHour': row['StartHour'],
        'StartTime': row['StartTime'],
        'StartEpoch': row['StartEpoch'],
        'EndTime': row['EndTime'],
        'EndEpoch': row['EndEpoch'],
        'StartDay': row['StartDay'],
        'EndDay': row['EndDay'],
        'LateralRotation': row['LateralRotation'],
        'NumberOfPeaks': row['NumberOfPeaks']
    })

In [ ]:
customer_list = get_customer_list(KPIHub_Conn)
arguments = {'conn': KPIHub_Conn}
customer = customer_list.iloc[0]
surveyIngester = SurveySummaryIngester(arguments)
surveyIngester.set_customer_info(customer)
surveyIngester.update_check()
surveyIngester.query_data()
surveyIngester.push_data()
surveyIngester.sanity_check()